In [4]:
%load_ext sql

## Step 2 - Connect to the SQLite database
- `import sqlite3` brings in Python's built-in SQLite library. 
- `sqlite3.connect("MontclairUni.db)` opens (or creates) a database file named `MontclairUni.db` in the current folder. The returned `conn` object represents the connection. 
- `conn.cursor()` creates a *cursor* - the object we use to actually execute SQL statements and fetch results. 


In [5]:
import sqlite3
conn = sqlite3.connect("MontclairUni.db")
cur = conn.cursor()

## Step 3 - Drop existing tables (clean slate)

`DROP TABLE IF EXISTS` - This will help 

In [ ]:
cur.execute("DROP TABLE IF EXISTS Enrollments")
cur.execute("DROP TABLE IF EXISTS Students")
cur.execute("DROP TABLE IF EXISTS Courses")
conn.commit()
print("Old tables dropped (if they existed).")

In [6]:
cur.execute("""
CREATE TABLE Students (
    StudentID INTEGER PRIMARY KEY,
    FirstName TEXT NOT NULL,
    LastName  TEXT NOT NULL,
    Age       INTEGER
)
""")
conn.commit()
print("Students table created.")

SyntaxError: EOF while scanning triple-quoted string literal (1417494724.py, line 5)

In [ ]:
cur.execute("""
CREATE TABLE Courses (
    CourseID   INTEGER PRIMARY KEY,
    CourseName TEXT NOT NULL,
    Credits    INTEGER NOT NULL
)
""")
conn.commit()
print("Courses table created.")

In [ ]:
cur.execute("""
CREATE TABLE Enrollments (
    StudentID INTEGER,
    CourseID  INTEGER,
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),
    FOREIGN KEY (CourseID)  REFERENCES Courses(CourseID)
)
""")
conn.commit()
print("Enrollments table created.")

In [ ]:
students = [
    (1, "Alice",   "Johnson",  20),
    (2, "Bob",     "Smith",    22),
    (3, "Carlos",  "Diaz",     19),
    (4, "Diana",   "Patel",    21),
    (5, "Ethan",   "Williams", 23),
]
cur.executemany("INSERT INTO Students VALUES (?, ?, ?, ?)", students)
conn.commit()
print(f"Inserted {len(students)} students.")

In [ ]:
courses = [
    (101, "Database Systems",        3),
    (102, "Data Structures",         4),
    (103, "Operating Systems",       3),
    (104, "Calculus I",              4),
    (105, "Introduction to AI",      3),
    (106, "Computer Networks",       3),
]
cur.executemany("INSERT INTO Courses VALUES (?, ?, ?)", courses)
conn.commit()
print(f"Inserted {len(courses)} courses.")

In [ ]:
enrollments = [
    (1, 101), (1, 102), (1, 105),
    (2, 101), (2, 103),
    (3, 102), (3, 104),
    (4, 101), (4, 105), (4, 106),
    (5, 103), (5, 104),
]
cur.executemany("INSERT INTO Enrollments VALUES (?, ?)", enrollments)
conn.commit()
print(f"Inserted {len(enrollments)} enrollments.")

In [ ]:
print("--- Students ---")
for row in cur.execute("SELECT * FROM Students"):
    print(row)

print("\n--- Courses ---")
for row in cur.execute("SELECT * FROM Courses"):
    print(row)

print("\n--- Enrollments ---")
for row in cur.execute("SELECT * FROM Enrollments"):
    print(row)

In [ ]:
def list_courses():
    cur.execute("SELECT CourseID, CourseName, Credits FROM Courses ORDER BY CourseID")
    rows = cur.fetchall()
    print("\nAll Courses:")
    print(f"  {'ID':<6}{'Name':<30}{'Credits'}")
    print("  " + "-" * 45)
    for cid, cname, credits in rows:
        print(f"  {cid:<6}{cname:<30}{credits}")


def enroll(student_id):
    try:
        cid = int(input("Enter Course ID to enroll in: "))
    except ValueError:
        print("Invalid Course ID.")
        return
    # Check the course exists
    cur.execute("SELECT CourseName FROM Courses WHERE CourseID = ?", (cid,))
    course = cur.fetchone()
    if not course:
        print(f"No course with ID {cid}.")
        return
    # Conflict check: already enrolled?
    cur.execute(
        "SELECT 1 FROM Enrollments WHERE StudentID = ? AND CourseID = ?",
        (student_id, cid),
    )
    if cur.fetchone():
        print(f"Already enrolled in {course[0]}. Cannot enroll twice.")
        return
    cur.execute("INSERT INTO Enrollments VALUES (?, ?)", (student_id, cid))
    conn.commit()
    print(f"Successfully enrolled in {course[0]}.")


def withdraw(student_id):
    try:
        cid = int(input("Enter Course ID to withdraw from: "))
    except ValueError:
        print("Invalid Course ID.")
        return
    cur.execute(
        "SELECT 1 FROM Enrollments WHERE StudentID = ? AND CourseID = ?",
        (student_id, cid),
    )
    if not cur.fetchone():
        print("You are not enrolled in that course.")
        return
    cur.execute(
        "DELETE FROM Enrollments WHERE StudentID = ? AND CourseID = ?",
        (student_id, cid),
    )
    conn.commit()
    print(f"Withdrawn from course {cid}.")


def search_course():
    term = input("Enter part of the course name to search: ").strip()
    if not term:
        print("Empty search term.")
        return
    cur.execute(
        "SELECT CourseID, CourseName, Credits FROM Courses WHERE CourseName LIKE ?",
        ("%" + term + "%",),
    )
    rows = cur.fetchall()
    if not rows:
        print(f"No courses matched \"{term}\".")
        return
    print(f"\nCourses matching \"{term}\":")
    for cid, cname, credits in rows:
        print(f"  {cid} - {cname} ({credits} credits)")


def my_classes(student_id):
    cur.execute(
        """
        SELECT c.CourseID, c.CourseName, c.Credits
        FROM Courses c
        JOIN Enrollments e ON c.CourseID = e.CourseID
        WHERE e.StudentID = ?
        ORDER BY c.CourseID
        """,
        (student_id,),
    )
    rows = cur.fetchall()
    if not rows:
        print("You are not enrolled in any classes yet.")
        return
    print("\nMy Classes:")
    for cid, cname, credits in rows:
        print(f"  {cid} - {cname} ({credits} credits)")


def get_active_student():
    """Ask for a student ID; -1 creates a new one. Returns the active StudentID."""
    while True:
        raw = input("\nEnter Student ID (or -1 to create a new student): ").strip()
        try:
            sid = int(raw)
        except ValueError:
            print("Please enter a number.")
            continue

        if sid == -1:
            # Create a new student
            try:
                new_id = int(input("  New Student ID: "))
            except ValueError:
                print("  Invalid ID.")
                continue
            cur.execute("SELECT 1 FROM Students WHERE StudentID = ?", (new_id,))
            if cur.fetchone():
                print(f"  ID {new_id} already exists. Try again.")
                continue
            first = input("  First name: ").strip()
            last  = input("  Last name: ").strip()
            try:
                age = int(input("  Age: "))
            except ValueError:
                print("  Invalid age.")
                continue
            cur.execute(
                "INSERT INTO Students VALUES (?, ?, ?, ?)",
                (new_id, first, last, age),
            )
            conn.commit()
            print(f"  Created student {first} {last} (ID {new_id}).")
            return new_id

        # Otherwise, look the student up
        cur.execute(
            "SELECT FirstName, LastName FROM Students WHERE StudentID = ?", (sid,)
        )
        row = cur.fetchone()
        if row:
            print(f"Welcome, {row[0]} {row[1]}!")
            return sid
        print(f"No student found with ID {sid}.")


def main():
    print("=" * 50)
    print("Montclair University Enrollment System")
    print("=" * 50)
    student_id = get_active_student()

    while True:
        print("\n--- Main Menu ---")
        print("  L - List all courses")
        print("  E - Enroll in a course")
        print("  W - Withdraw from a course")
        print("  S - Search course by name")
        print("  M - My classes")
        print("  X - Exit")
        choice = input("Choose an option: ").strip().upper()

        if choice == "L":
            list_courses()
        elif choice == "E":
            enroll(student_id)
        elif choice == "W":
            withdraw(student_id)
        elif choice == "S":
            search_course()
        elif choice == "M":
            my_classes(student_id)
        elif choice == "X":
            print("Goodbye!")
            break
        else:
            print("Invalid option. Pick L, E, W, S, M, or X.")


# Run the application:
main()

In [ ]:
conn.close()
print("Connection closed. MontclairUni.db is saved in the current folder.")